# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (`@id`s), fields, and columns as defined by the Croissant metadata. All entities are referenced by `@id`.

In [ ]:
# List all record set @id's
record_sets = [rs['@id'] for rs in metadata.record_sets]
print("Available Record Sets (by @id):\n")
for rs in metadata.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'No name')}")

# For each record set, list its fields (and columns if available)
for rs in metadata.record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    print("  Fields:")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_id = field.get('@id', str(field))
        f_name = field.get('name', '-')
        print(f"    - {field_id}: {f_name}")

    # List columns if present
    if 'column' in rs:
        columns = rs['column']
        if isinstance(columns, dict):
            columns = [columns]
        print("  Columns:")
        for col in columns:
            col_id = col.get('@id', str(col))
            c_name = col.get('name', '-')
            print(f"    - {col_id}: {c_name}")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. All record sets are referenced by their `@id` from the overview above.

In [ ]:
# Extract data from all record sets using their @id
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from Record Set: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), "\n")
    except Exception as e:
        print(f"Could not load data from Record Set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply exploratory data analysis to a record set and field. We'll select a record set that contains numerical health scores, such as GAD-7, PHQ-9, or PSQ, if present.

In [ ]:
# Try to identify a record set with GAD-7 or PHQ-9 or PSQ score field
# (The actual field names/ids may need to be checked above.)

# This example assumes the main survey data is in the first record set
record_set_id = record_sets[0]  # Update this if you want a different record set
df = dataframes[record_set_id]

# Try to identify suitable numeric fields
potential_numeric_fields = [col for col in df.columns if any(keyword in col.lower() for keyword in ["score", "gad", "phq", "psq"]) or df[col].dtype in ['int64', 'float64']]
print(f"Potential numeric fields: {potential_numeric_fields}")
if potential_numeric_fields:
    numeric_field = potential_numeric_fields[0]
    print(f"Using numeric field: {numeric_field} for EDA.")
    
    threshold = df[numeric_field].mean() if df[numeric_field].dtype in ['float64', 'int64'] else 10
    
    # Filtering for values greater than the threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    
    # Normalizing the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by a demographic field
    group_candidates = [col for col in df.columns if any(x in col.lower() for x in ["gender", "sex", "age", "education", "village", "marital"])]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df)
    else:
        print("No suitable grouping field found.")
else:
    print("No numeric fields found suitable for EDA.")

## 5. Visualization
Visualize data distributions or relationships in the main record set using matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field if available
if 'numeric_field' in locals():
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field exists, boxplot by group
    if 'group_field' in locals():
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and process a Croissant-based dataset using `mlcroissant`. Key findings depend on the field explorations (e.g., mean GAD-7/PHQ-9 scores, demographic differences, etc.). Further analysis can build on this foundation for public health research, statistical summaries, or AI model training.

**Note:** For precise semantic meanings, always consult the Croissant metadata field and column documentation via their `@id` in the schema.